# Delivery Delay Analysis

## Project Overview
Analyze order delivery data to identify causes of late deliveries by route, warehouse, carrier, and time period. This is a descriptive analytics project.

## Learning Objectives
- Calculate on-time delivery rates across multiple dimensions
- Identify the leading causes of delivery delays
- Detect seasonal and geographic delay patterns
- Produce actionable recommendations for logistics improvement

## Problem Statement
An e-commerce company is experiencing rising late delivery complaints. Management needs to understand where delays originate — specific warehouses, carriers, routes, or time periods — to target interventions.

## Why This Project Matters
Late deliveries erode customer trust, increase support costs, and drive churn. Root-cause analysis enables targeted fixes rather than blanket spending on logistics.

## Dataset Overview
Synthetic order delivery data: ~8K orders over 2 years with warehouse, carrier, route, and timing details.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 8000
warehouses = np.random.choice(['WH-East', 'WH-West', 'WH-Central', 'WH-South'], n, p=[0.30, 0.25, 0.25, 0.20])
carriers = np.random.choice(['FastShip', 'EcoFreight', 'PrimeLogistics', 'BudgetCarrier'], n, p=[0.30, 0.25, 0.25, 0.20])
regions = np.random.choice(['Northeast', 'Southeast', 'Midwest', 'West', 'Southwest'], n, p=[0.22, 0.20, 0.18, 0.25, 0.15])
order_types = np.random.choice(['Standard', 'Express', 'Same-Day'], n, p=[0.55, 0.30, 0.15])

dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
order_dates = np.array([pd.Timestamp(d) for d in np.random.choice(dates, n)])

# Promised days by order type
promised = {'Standard': 5, 'Express': 2, 'Same-Day': 1}
promised_days = np.array([promised[t] for t in order_types])

# Actual delivery: base + warehouse effect + carrier effect + noise
wh_delay = {'WH-East': 0, 'WH-West': 0.5, 'WH-Central': -0.3, 'WH-South': 1.0}
cr_delay = {'FastShip': -0.5, 'EcoFreight': 0.8, 'PrimeLogistics': 0, 'BudgetCarrier': 1.5}
actual_days = np.array([
    max(1, int(promised[t] + wh_delay[w] + cr_delay[c] + np.random.normal(0.5, 1.5)))
    for t, w, c in zip(order_types, warehouses, carriers)
])

delay_days = np.maximum(0, actual_days - promised_days)
on_time = (delay_days == 0).astype(int)

df = pd.DataFrame({
    'OrderID': [f'ORD{i:06d}' for i in range(n)],
    'OrderDate': order_dates,
    'Warehouse': warehouses, 'Carrier': carriers,
    'Region': regions, 'OrderType': order_types,
    'PromisedDays': promised_days, 'ActualDays': actual_days,
    'DelayDays': delay_days, 'OnTime': on_time,
})
df['YearMonth'] = df['OrderDate'].dt.to_period('M')
df['DayOfWeek'] = df['OrderDate'].dt.day_name()

print(f'Dataset shape: {df.shape}')
print(f'On-time rate: {df["OnTime"].mean():.1%}')
print(f'Avg delay (when late): {df.loc[df["DelayDays"]>0, "DelayDays"].mean():.1f} days')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["OrderDate"].min().date()} to {df["OrderDate"].max().date()}')
print(f'\nOrder type distribution:\n{df["OrderType"].value_counts()}')
print(f'\nCarrier distribution:\n{df["Carrier"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Carrier')['OnTime'].mean().sort_values().plot.barh(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('On-Time Rate by Carrier')

df.groupby('Warehouse')['OnTime'].mean().sort_values().plot.barh(ax=axes[0,1], edgecolor='black', color='coral')
axes[0,1].set_title('On-Time Rate by Warehouse')

monthly_otd = df.groupby('YearMonth')['OnTime'].mean()
monthly_otd.plot(ax=axes[1,0], marker='o', color='green')
axes[1,0].set_title('Monthly On-Time Delivery Rate')
axes[1,0].tick_params(axis='x', rotation=45)

df['DelayDays'].hist(bins=range(0, 15), ax=axes[1,1], edgecolor='black', alpha=0.7, color='mediumpurple')
axes[1,1].set_title('Delay Days Distribution')
axes[1,1].set_xlabel('Days Late')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Delay by Order Type

In [ ]:
type_stats = df.groupby('OrderType').agg(
    orders=('OrderID', 'count'),
    on_time_rate=('OnTime', 'mean'),
    avg_delay=('DelayDays', 'mean'),
    avg_actual=('ActualDays', 'mean'),
    avg_promised=('PromisedDays', 'mean'),
).round(3)
print('Delivery Performance by Order Type:')
print(type_stats)

fig, ax = plt.subplots(figsize=(8, 5))
type_stats['on_time_rate'].plot.bar(ax=ax, edgecolor='black', color=['steelblue','coral','green'])
ax.set_title('On-Time Rate by Order Type')
ax.set_ylabel('On-Time Rate')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'order_type.png'), dpi=100, bbox_inches='tight')
plt.show()

## Carrier × Warehouse Performance

In [ ]:
cross = df.groupby(['Carrier', 'Warehouse'])['OnTime'].mean().unstack().round(3)
print('On-Time Rate — Carrier × Warehouse:')
print(cross)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(cross, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, vmin=0.3, vmax=0.9)
ax.set_title('On-Time Rate: Carrier × Warehouse')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'carrier_warehouse.png'), dpi=100, bbox_inches='tight')
plt.show()

## Regional Delay Patterns

In [ ]:
reg_stats = df.groupby('Region').agg(
    orders=('OrderID', 'count'),
    on_time_rate=('OnTime', 'mean'),
    avg_delay=('DelayDays', 'mean'),
).round(3).sort_values('on_time_rate')
print('Regional Performance:')
print(reg_stats)

fig, ax = plt.subplots(figsize=(10, 5))
reg_stats['on_time_rate'].plot.barh(ax=ax, edgecolor='black', color='teal')
ax.set_title('On-Time Rate by Region')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'regional.png'), dpi=100, bbox_inches='tight')
plt.show()

## Day-of-Week Effect

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df.groupby('DayOfWeek')['OnTime'].mean().reindex(dow_order)
fig, ax = plt.subplots(figsize=(10, 5))
dow.plot.bar(ax=ax, edgecolor='black', color='orange')
ax.set_title('On-Time Rate by Day of Week (Order Date)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'day_of_week.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **BudgetCarrier** has the worst on-time rate — cost savings may not justify the service quality gap
- **WH-South** underperforms other warehouses — investigate staffing, processes, or capacity constraints
- **Same-Day** orders have the lowest on-time rate due to tight SLA — consider adjusting promises or adding capacity
- **Carrier × Warehouse** heatmap reveals specific combinations that are particularly problematic
- **Monthly trends** can reveal seasonal bottlenecks (e.g., holiday surges) and capacity planning needs

## Limitations
- Synthetic data with fixed carrier/warehouse effects — real logistics is more dynamic
- No cost data (shipping cost, penalty costs for late delivery)
- No weather, traffic, or external disruption data
- No package-level details (size, weight, fragility)
- No customer satisfaction linkage

## How to Improve This Project
- Use real order and shipment data
- Add cost-of-delay analysis (refunds, re-ships, churn)
- Include external factors (weather, holidays, disruptions)
- Build predictive delay models for proactive customer communication
- Add last-mile delivery analysis with geospatial data

## Production Considerations
- Real-time delay prediction for proactive customer notifications
- Carrier scorecards updated weekly for contract negotiations
- Capacity planning dashboards by warehouse and season
- Automated routing optimization based on historical performance

## Common Mistakes
- Using average delay instead of on-time rate (masks bimodal distributions)
- Not controlling for order type when comparing carriers
- Ignoring the interaction between warehouse and carrier
- Treating all delays equally regardless of severity

## Mini Challenge / Exercises
1. Which carrier-warehouse pair has the worst on-time rate? How many orders does it handle?
2. What % of all late orders come from the worst-performing carrier? Is it disproportionate?
3. Calculate the 95th percentile delivery time for each order type. How does it compare to the promise?
4. If BudgetCarrier's on-time rate matched FastShip's, what would the overall on-time rate become?

## Final Summary / Key Takeaways
- Delivery delay analysis reveals specific operational weak points, not just overall metrics
- Carrier and warehouse performance vary significantly — targeted improvements beat blanket spending
- Order type SLA feasibility should be validated against actual performance
- Cross-dimensional analysis (carrier × warehouse, region × type) uncovers hidden patterns
- Continuous monitoring enables proactive logistics management